<a href="https://colab.research.google.com/github/bru02/epstein/blob/main/notebooks/corpus_analysis_linguistic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Corpus analysis with spaCy linguistics

Run in Colab or locally. This copy uses spaCy lemmatization, POS filtering, and named entities before sklearn vectorization.


In [ ]:
is_colab = "google.colab" in str(get_ipython())
if is_colab:
    get_ipython().system("uv pip install --system 'epstein @ git+https://github.com/bru02/epstein'")

In [ ]:
%matplotlib inline
from collections import Counter
import json
from pathlib import Path
import re
from time import perf_counter

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import spacy
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS, TfidfVectorizer

Path("outputs_linguistic").mkdir(exist_ok=True)
started_at = perf_counter()
last_at = started_at
token_pattern = re.compile(r"\b[a-zA-Z][a-zA-Z']{2,}\b")
sentence_pattern = re.compile(r"[.!?]+")
stop_words = set(ENGLISH_STOP_WORDS)

# Set to None for the full corpus. Use a smaller number while iterating.
sample_documents = 50_000
sample_seed = 7
spacy_model = "en_core_web_sm"
spacy_batch_size = 1_000
allowed_pos = {"NOUN", "PROPN", "VERB", "ADJ"}
keep_named_entities = True
entity_limit_per_document = 20
vectorizer_min_df = 5
vectorizer_max_df = 0.8
vectorizer_ngram_range = (1, 3)
vectorizer_max_features = None


def log(label):
    global last_at
    now = perf_counter()
    print(f"[analysis] {label}: phase={now - last_at:.2f}s total={now - started_at:.2f}s", flush=True)
    last_at = now


def write_frame(frame, path):
    frame.write_csv(Path("outputs_linguistic") / path)
    print(f"[analysis] wrote outputs_linguistic/{path}", flush=True)


def load_spacy_model(enable_entities=True):
    disabled = ["parser"]
    if not enable_entities:
        disabled.append("ner")
    try:
        return spacy.load(spacy_model, disable=disabled)
    except OSError:
        import en_core_web_sm
        return en_core_web_sm.load(disable=disabled)


def linguistic_tokens(doc):
    tokens = []
    for token in doc:
        lemma = token.lemma_.lower().strip()
        if token.is_space or token.is_punct or token.like_num:
            continue
        if token.pos_ not in allowed_pos:
            continue
        if lemma in stop_words or len(lemma) < 3:
            continue
        if not token_pattern.fullmatch(lemma):
            continue
        tokens.append(lemma)
    return tokens


def count_syllables(word):
    cleaned = re.sub(r"[^a-z]", "", word.lower())
    if not cleaned:
        return 0
    count = len(re.findall(r"[aeiouy]+", cleaned))
    if cleaned.endswith("e") and count > 1:
        count -= 1
    return max(1, count)


def text_metrics(text, sentence_count=None):
    raw_tokens = token_pattern.findall(text or "")
    if sentence_count is None:
        sentence_count = len([part for part in sentence_pattern.split(text or "") if part.strip()])
    if not raw_tokens or sentence_count == 0:
        return None
    syllables = sum(count_syllables(token) for token in raw_tokens)
    words = len(raw_tokens)
    types = len(set(token.lower() for token in raw_tokens))
    return {
        "tokens": words,
        "sentences": sentence_count,
        "types": types,
        "type_token_ratio": types / words,
        "flesch_reading_ease": 206.835 - 1.015 * (words / sentence_count) - 84.6 * (syllables / words),
        "flesch_kincaid_grade": 0.39 * (words / sentence_count) + 11.8 * (syllables / words) - 15.59,
    }


def plot_barh(labels, values, path, title, xlabel, color):
    plt.figure(figsize=(9, 7))
    plt.barh(labels, values, color=color)
    plt.xlabel(xlabel)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(Path("outputs_linguistic") / path, dpi=160)
    plt.show()


def compact_language_text(text):
    compact = re.sub(r"\s+", " ", text or "").strip()
    return compact.replace("\n", " ")


def load_language_model():
    try:
        import fasttext
        model_path = Path("models") / "lid.176.ftz"
        model_path.parent.mkdir(exist_ok=True)
        if not model_path.exists():
            import urllib.request
            urllib.request.urlretrieve(
                "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz",
                model_path,
            )
        return fasttext.load_model(str(model_path))
    except Exception as error:
        print(f"[analysis] fastText language detection unavailable: {error}", flush=True)
        return None


def fallback_language_indicator(text):
    compact = compact_language_text(text)
    if len(compact) < 80:
        return "too_short"
    raw_tokens = [match.group(0).lower() for match in token_pattern.finditer(compact[:1000])]
    if sum(1 for token in raw_tokens[:200] if token in stop_words) >= 3:
        return "likely_en"
    if compact.isascii():
        return "unknown_ascii"
    return "unknown_non_ascii"


def flush_language_batch(language_model, language_batch, language_counter, confidence_counter):
    if not language_batch:
        return
    labels, probabilities = language_model.predict(language_batch, k=1)
    for label_values, probability_values in zip(labels, probabilities):
        label = label_values[0].replace("__label__", "")
        language_counter[label] += 1
        confidence_counter[label] += float(probability_values[0])
    language_batch.clear()


## Load data

In [ ]:
data_base = "data" if Path("data/fact_emails.parquet").exists() else "https://github.com/bru02/epstein/raw/main/data"

emails = pl.read_parquet(f"{data_base}/fact_emails.parquet")
bridge = pl.read_parquet(f"{data_base}/bridge_email_people.parquet")
people = pl.read_parquet(f"{data_base}/dim_people.parquet")
analysis_filter = pl.col("text_token_len") > 0
analysis_emails = emails.filter(analysis_filter)
valid_dated_emails = emails.filter(pl.col("year").is_not_null())
if sample_documents is None:
    sampled_analysis_emails = analysis_emails
else:
    sampled_analysis_emails = analysis_emails.sample(n=min(sample_documents, analysis_emails.height), seed=sample_seed)
log("loaded parquet")

print(f"raw documents: {emails.height:,}")
print(f"analysis documents: {analysis_emails.height:,}")
print(f"sampled analysis documents: {sampled_analysis_emails.height:,}")


## Corpus statistics

In [ ]:

summary_rows = [
    ("raw_documents", f"{emails.height}"),
    ("analysis_documents", f"{analysis_emails.height}"),
    ("sampled_analysis_documents", f"{sampled_analysis_emails.height}"),
    ("average_document_tokens", f"{float(emails.get_column('text_token_len').mean()):.2f}"),
    ("median_document_tokens", f"{float(emails.get_column('text_token_len').median()):.0f}"),
    ("average_document_characters", f"{float(emails.get_column('text_char_len').mean()):.2f}"),
    ("median_document_characters", f"{float(emails.get_column('text_char_len').median()):.0f}"),
    ("empty_text_rows", f"{emails.filter(pl.col('text_token_len') == 0).height}"),
    ("very_short_rows_under_10_tokens", f"{emails.filter(pl.col('text_token_len') < 10).height}"),
    ("sender_redacted_rows", f"{emails.filter(pl.col('sender_redacted')).height}"),
    ("raw_min_sent_at", f"{emails.get_column('sent_at').min()}"),
    ("raw_max_sent_at", f"{emails.get_column('sent_at').max()}"),
    ("valid_dated_rows_1990_2019", f"{valid_dated_emails.height}"),
    ("missing_or_out_of_range_date_rows", f"{emails.height - valid_dated_emails.height}"),
    ("min_valid_sent_at", f"{valid_dated_emails.get_column('sent_at').min()}"),
    ("max_valid_sent_at", f"{valid_dated_emails.get_column('sent_at').max()}"),
    ("distinct_valid_years", f"{valid_dated_emails.get_column('year').n_unique()}"),
    ("people_rows", f"{people.height}"),
    ("email_person_links", f"{bridge.height}"),
]
write_frame(pl.DataFrame(summary_rows, schema=["metric", "value"], orient="row"), "corpus_summary.csv")

missing_exprs = []
for column in emails.columns:
    expr = pl.col(column).is_null()
    if emails.schema[column] == pl.String:
        expr = expr | (pl.col(column).str.strip_chars() == "")
    missing_exprs.append(expr.mean().alias(column))
missingness = (
    emails.select(missing_exprs)
    .transpose(include_header=True, header_name="field", column_names=["missing_rate"])
    .with_columns((pl.col("missing_rate") * 100).round(2).alias("missing_pct"))
    .sort("missing_rate", descending=True)
)
write_frame(missingness, "metadata_missingness.csv")

display(pl.DataFrame(summary_rows, schema=["metric", "value"], orient="row"))
display(missingness)

missing_plot = missingness.sort("missing_rate")
plt.figure(figsize=(8, 6))
plt.barh(missing_plot["field"], missing_plot["missing_pct"], color="#4c78a8")
plt.xlabel("Missing values (%)")
plt.title("Metadata missingness")
plt.tight_layout()
plt.savefig(Path("outputs_linguistic") / "metadata_missingness.png", dpi=160)
plt.show()


## Date anomalies and time span

In [ ]:
date_anomalies = (
    emails.filter(pl.col("sent_at").is_not_null() & pl.col("year").is_null())
    .select("email_id", "doc_id", "subject", "sent_at", "release_batch", "folder_path", "text_token_len")
    .sort("sent_at")
)
missing_sent_at_rows = emails.filter(pl.col("sent_at").is_null()).height
write_frame(
    pl.DataFrame(
        [
            ("missing_sent_at_rows", f"{missing_sent_at_rows}"),
            ("parsed_but_out_of_range_year_rows", f"{date_anomalies.height}"),
            ("missing_or_out_of_range_date_rows", f"{missing_sent_at_rows + date_anomalies.height}"),
            ("earliest_raw_sent_at", f"{emails.get_column('sent_at').min()}"),
            ("latest_raw_sent_at", f"{emails.get_column('sent_at').max()}"),
        ],
        schema=["metric", "value"],
        orient="row",
    ),
    "date_anomaly_summary.csv",
)
write_frame(date_anomalies.head(1000), "date_anomalies_sample.csv")

by_year = emails.group_by("year").agg(pl.len().alias("documents")).sort("year")
release_batch_coverage = emails.group_by("release_batch").agg(pl.len().alias("documents")).sort("release_batch")
write_frame(by_year, "documents_by_year.csv")
write_frame(release_batch_coverage, "release_batch_coverage.csv")
log("corpus metadata")

display(by_year)
display(release_batch_coverage)

## spaCy linguistic pass, language detection, readability, and KWIC

Controlled document loop over the sampled analysis set. spaCy produces lemma/POS-filtered texts and named entities; sklearn vectorizers consume the lemma strings in the next section.

In [ ]:
readability_rows = []
language_counter = Counter()
language_confidence = Counter()
language_model = load_language_model()
language_batch = []
entity_rows = []
kwic_rows = {term: [] for term in ["island", "flight", "passport", "massage", "Maxwell"]}
kwic_patterns = {term: re.compile(re.escape(term), re.IGNORECASE) for term in kwic_rows}
doc_count = sampled_analysis_emails.height
analysis_records = sampled_analysis_emails.select("email_id", "sent_at", "subject", "text_clean").to_dicts()
analysis_texts = [(row["text_clean"] or "") for row in analysis_records]
linguistic_texts = []
nlp = load_spacy_model(enable_entities=keep_named_entities)

for index, (row, doc) in enumerate(zip(analysis_records, nlp.pipe(analysis_texts, batch_size=spacy_batch_size)), start=1):
    text = row["text_clean"] or ""
    language_text = compact_language_text(text)
    if len(language_text) < 80:
        language_counter["too_short"] += 1
    elif language_model is None:
        language_counter[fallback_language_indicator(text)] += 1
    else:
        language_batch.append(language_text[:1000])
        if len(language_batch) >= 4096:
            flush_language_batch(language_model, language_batch, language_counter, language_confidence)

    tokens = linguistic_tokens(doc)
    linguistic_texts.append(" ".join(tokens))
    if len(tokens) >= 50:
        metrics = text_metrics(text, sentence_count=len(list(doc.sents)) if doc.has_annotation("SENT_START") else None)
        if metrics:
            readability_rows.append(metrics)

    if keep_named_entities:
        for entity in list(doc.ents)[:entity_limit_per_document]:
            entity_text = compact_language_text(entity.text)
            if entity_text:
                entity_rows.append(
                    {
                        "email_id": row["email_id"],
                        "sent_at": row["sent_at"],
                        "subject": row["subject"],
                        "entity": entity_text,
                        "label": entity.label_,
                    }
                )

    for term, pattern in kwic_patterns.items():
        if len(kwic_rows[term]) >= 500:
            continue
        for match in pattern.finditer(text):
            kwic_rows[term].append(
                {
                    "email_id": row["email_id"],
                    "sent_at": row["sent_at"],
                    "subject": row["subject"],
                    "left": text[max(0, match.start() - 80) : match.start()].replace("\n", " "),
                    "keyword": text[match.start() : match.end()],
                    "right": text[match.end() : match.end() + 80].replace("\n", " "),
                }
            )
            if len(kwic_rows[term]) >= 500:
                break
    if index % 10_000 == 0:
        print(f"[analysis] linguistically processed {index:,}", flush=True)

if language_model is not None:
    flush_language_batch(language_model, language_batch, language_counter, language_confidence)
log("spacy lemmas, entities, language indicators, readability, kwic")


## Language distribution

In [ ]:
language_distribution = (
    pl.DataFrame(
        [
            (
                language,
                documents,
                None if language_confidence[language] == 0 else round(language_confidence[language] / documents, 4),
            )
            for language, documents in language_counter.items()
        ],
        schema=["language", "documents", "avg_confidence"],
        orient="row",
    )
    .with_columns((pl.col("documents") / pl.col("documents").sum() * 100).round(2).alias("pct"))
    .sort("documents", descending=True)
)
write_frame(language_distribution, "language_distribution.csv")
display(language_distribution)

## Term and n-gram frequencies

In [ ]:
def vectorizer_table(matrix, feature_names, label):
    freqs = matrix.sum(axis=0).A1
    doc_freqs = matrix.getnnz(axis=0)
    return pl.DataFrame(
        {
            label: feature_names,
            "frequency": freqs,
            "document_frequency": doc_freqs,
            "document_frequency_pct": doc_freqs / doc_count * 100,
        }
    ).sort("frequency", descending=True)


term_vectorizer = CountVectorizer(
    token_pattern=r"(?u)\b[a-z][a-z']{2,}\b",
    min_df=vectorizer_min_df,
    max_df=vectorizer_max_df,
    ngram_range=(1, 1),
    max_features=vectorizer_max_features,
    dtype=np.int32,
)
term_matrix = term_vectorizer.fit_transform(linguistic_texts)
term_frequency_table = vectorizer_table(term_matrix, term_vectorizer.get_feature_names_out(), "term")
term_frequency_table.head(100_000).write_csv(Path("outputs_linguistic") / "term_frequencies.csv")
top_terms = term_frequency_table.head(25).reverse()
plot_barh(top_terms["term"], top_terms["frequency"], "top_terms.png", "Top lemmatized unigram frequencies", "Frequency", "#72b7b2")

ngram_vectorizer = CountVectorizer(
    token_pattern=r"(?u)\b[a-z][a-z']{2,}\b",
    min_df=vectorizer_min_df,
    max_df=vectorizer_max_df,
    ngram_range=(2, vectorizer_ngram_range[1]),
    max_features=vectorizer_max_features,
    dtype=np.int32,
)
ngram_matrix = ngram_vectorizer.fit_transform(linguistic_texts)
ngram_table = vectorizer_table(ngram_matrix, ngram_vectorizer.get_feature_names_out(), "ngram")
ngram_table.head(100_000).write_csv(Path("outputs_linguistic") / "ngrams.csv")
top_ngrams = ngram_table.head(25).reverse()
plot_barh(top_ngrams["ngram"], top_ngrams["frequency"], "top_ngrams.png", "Top lemmatized n-grams", "Frequency", "#b279a2")
log("sklearn count vectorizers over spaCy lemmas")

tfidf_vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\b[a-z][a-z']{2,}\b",
    min_df=vectorizer_min_df,
    max_df=vectorizer_max_df,
    ngram_range=vectorizer_ngram_range,
    max_features=vectorizer_max_features,
    dtype=np.float32,
)
tfidf_matrix = tfidf_vectorizer.fit_transform(linguistic_texts)
tfidf_table = pl.DataFrame(
    {
        "term": tfidf_vectorizer.get_feature_names_out(),
        "tfidf_keyness": tfidf_matrix.sum(axis=0).A1,
        "document_frequency": tfidf_matrix.getnnz(axis=0),
    }
).with_columns(
    (pl.col("document_frequency") / doc_count * 100).alias("document_frequency_pct")
).sort("tfidf_keyness", descending=True)
tfidf_table.head(100_000).write_csv(Path("outputs_linguistic") / "tfidf_key_terms.csv")
top_tfidf = tfidf_table.head(25).reverse()
plot_barh(top_tfidf["term"], top_tfidf["tfidf_keyness"], "top_tfidf_terms.png", "Top lemmatized TF-IDF terms", "Summed TF-IDF", "#ff9da6")
log("sklearn tf-idf over spaCy lemmas")


## Named entities

In [ ]:
entity_table = pl.DataFrame(entity_rows)
if entity_table.is_empty():
    entity_summary = pl.DataFrame(schema={"entity": pl.String, "label": pl.String, "frequency": pl.UInt32, "document_frequency": pl.UInt32})
else:
    entity_summary = (
        entity_table.group_by("entity", "label")
        .agg(pl.len().alias("frequency"), pl.col("email_id").n_unique().alias("document_frequency"))
        .sort(["frequency", "document_frequency"], descending=True)
    )
entity_table.write_csv(Path("outputs_linguistic") / "entities.csv")
entity_summary.head(100_000).write_csv(Path("outputs_linguistic") / "entity_summary.csv")
display(entity_summary.head(50))

if not entity_summary.is_empty():
    top_entities = entity_summary.head(25).reverse()
    plot_barh(top_entities["entity"], top_entities["frequency"], "top_entities.png", "Top named entities", "Frequency", "#59a14f")
log("named entities")


## KWIC concordances

In [ ]:
for term, rows in kwic_rows.items():
    pl.DataFrame(rows).write_csv(Path("outputs_linguistic") / f"kwic_{term.lower()}.csv")
    print(f"kwic_{term.lower()}: {len(rows)} rows")
log("kwic")

## Readability and lexical diversity

In [ ]:
readability_table = pl.DataFrame(readability_rows)
readability_summary = readability_table.describe()
display(readability_summary)
readability_summary.write_csv(Path("outputs_linguistic") / "readability_lexical_summary.csv")
readability_table.sample(n=min(100_000, readability_table.height), seed=7).write_csv(
    Path("outputs_linguistic") / "readability_lexical_sample.csv"
)

plt.figure(figsize=(8, 4))
plt.hist(readability_table["type_token_ratio"].to_list(), bins=50, color="#9d755d")
plt.xlabel("Type-token ratio")
plt.ylabel("Documents")
plt.title("Lexical diversity")
plt.tight_layout()
plt.savefig(Path("outputs_linguistic") / "lexical_diversity.png", dpi=160)
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(readability_table["flesch_reading_ease"].to_list(), bins=50, color="#bab0ac")
plt.xlabel("Flesch reading ease")
plt.ylabel("Documents")
plt.title("Readability")
plt.tight_layout()
plt.savefig(Path("outputs_linguistic") / "readability.png", dpi=160)
plt.show()
log("readability")

## People network

In [ ]:
top_people = (
    bridge.group_by("person_id", "person_name")
    .agg(pl.col("email_id").n_unique().alias("documents"), pl.len().alias("links"))
    .sort("documents", descending=True)
    .head(100)
)
role_counts = (
    bridge.group_by("involvement_role")
    .agg(pl.len().alias("links"), pl.col("email_id").n_unique().alias("documents"))
    .sort("links", descending=True)
)
linked_document_count = bridge.select(pl.col("email_id").n_unique()).item()
people_coverage = pl.DataFrame(
    [
        ("documents_with_people", f"{linked_document_count}"),
        ("analysis_documents", f"{analysis_emails.height}"),
        ("documents_with_people_pct", f"{linked_document_count / analysis_emails.height * 100:.2f}"),
    ],
    schema=["metric", "value"],
    orient="row",
)
write_frame(top_people, "top_people.csv")
write_frame(role_counts, "involvement_role_counts.csv")
write_frame(people_coverage, "people_coverage.csv")

display(people_coverage)
display(role_counts)

top_people_plot = top_people.head(25).reverse()
plot_barh(top_people_plot["person_name"], top_people_plot["documents"], "top_people.png", "Top linked people", "Linked documents", "#59a14f")
log("people")

## Run metadata

In [ ]:
metadata = {
    "documents_available": analysis_emails.height,
    "documents_processed": doc_count,
    "sample_documents": sample_documents,
    "sample_seed": sample_seed,
    "spacy_model": spacy_model,
    "spacy_batch_size": spacy_batch_size,
    "allowed_pos": sorted(allowed_pos),
    "keep_named_entities": keep_named_entities,
    "entity_limit_per_document": entity_limit_per_document,
    "vectorizer_min_df": vectorizer_min_df,
    "vectorizer_max_df": vectorizer_max_df,
    "vectorizer_ngram_range": vectorizer_ngram_range,
    "vectorizer_max_features": vectorizer_max_features,
    "term_vectorizer_rows": term_frequency_table.height,
    "ngram_vectorizer_rows": ngram_table.height,
    "tfidf_vectorizer_rows": tfidf_table.height,
    "entity_rows": entity_table.height,
    "entity_summary_rows": entity_summary.height,
    "keyness_method": "spaCy lemma/POS-filtered texts passed to sklearn TfidfVectorizer, then summed document TF-IDF scores",
    "language_detection": "fastText lid.176.ftz when fasttext is installed; fallback buckets otherwise",
    "runtime_seconds": round(perf_counter() - started_at, 2),
}
(Path("outputs_linguistic") / "run_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")
print(json.dumps(metadata, indent=2))
